# 59 — Cluster Optimization Completes the Second Systems Arc

This notebook is a v2 synthesis for Notebook 59.  
It closes the second systems arc:

\[
43 \rightarrow 49 \rightarrow 53 \rightarrow 59
\]

The focus is cluster-level optimization: local adaptive serving becomes global adaptive serving through shared telemetry, bounded policy updates, replica placement, failure recovery, and capacity balancing.

In [ ]:
# Notebook 59 v2 setup
from pathlib import Path
import math
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch, Rectangle
from matplotlib.colors import ListedColormap, BoundaryNorm

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 28,
    "axes.labelsize": 18,
    "legend.fontsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

In [ ]:
# Shared drawing helpers

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=15, lw=2, radius=0.08):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.03,rounding_size={radius}",
        linewidth=lw,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, lw=2, style="-|>", connectionstyle="arc3,rad=0.0"):
    arr = FancyArrowPatch(
        start, end,
        arrowstyle=style,
        mutation_scale=16,
        linewidth=lw,
        color="black",
        connectionstyle=connectionstyle
    )
    ax.add_patch(arr)
    return arr

def draw_linear_pipeline(ax, labels, y=0.5, x0=0.05, x1=0.95, box_w=0.13, box_h=0.16, fontsize=14):
    n = len(labels)
    xs = np.linspace(x0, x1 - box_w, n)
    for i, (x, label) in enumerate(zip(xs, labels)):
        rounded_box(ax, (x, y), box_w, box_h, label, fontsize=fontsize)
        if i < n - 1:
            arrow(ax, (x + box_w, y + box_h/2), (xs[i+1], y + box_h/2))

## 1. Cluster optimization specifies global adaptive serving

The second arc closes when local scheduling decisions become coordinated cluster-level policy.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Cluster optimization specifies global adaptive serving", pad=24)

rounded_box(ax, (0.39, 0.76), 0.22, 0.11, "cluster\nobjective", fontsize=18)
rounded_box(ax, (0.35, 0.57), 0.30, 0.12, "global\noptimizer", fontsize=18)
arrow(ax, (0.50, 0.76), (0.50, 0.69), lw=2.4)

regions = [
    (0.10, 0.31, "region A\nscheduler", "serving\nloop A"),
    (0.39, 0.31, "region B\nscheduler", "serving\nloop B"),
    (0.68, 0.31, "region C\nscheduler", "serving\nloop C"),
]
for x, y, sched, loop in regions:
    rounded_box(ax, (x, y), 0.22, 0.12, sched, fontsize=15)
    rounded_box(ax, (x, y-0.19), 0.22, 0.12, loop, fontsize=15)
    arrow(ax, (x+0.11, 0.57), (x+0.11, y+0.12), connectionstyle="arc3,rad=0.1")
    arrow(ax, (x+0.11, y), (x+0.11, y-0.07))

rounded_box(ax, (0.31, 0.05), 0.38, 0.11, "shared telemetry fabric", fontsize=18)
for x, y, _, _ in regions:
    arrow(ax, (x+0.11, y-0.19), (0.50, 0.16), connectionstyle="arc3,rad=0.08")
arrow(ax, (0.50, 0.16), (0.50, 0.57), connectionstyle="arc3,rad=0.0")

ax.text(
    0.5, 0.005,
    "Local adaptation becomes cluster-wide optimization through bounded policy updates.",
    ha="center", va="bottom", fontsize=17
)

savefig("59_cluster_optimization_architecture.png")

## 2. Cluster operating-regime map

The optimizer chooses a cluster policy from reserve capacity and load imbalance.

In [ ]:
reserve = np.linspace(0.05, 0.95, 220)
imbalance = np.linspace(0.05, 0.95, 220)
R, I = np.meshgrid(reserve, imbalance)

# 0 steady, 1 reserve, 2 shed/shorten, 3 rebalance, 4 scale-out
policy = np.zeros_like(R, dtype=int)
policy[(R > 0.58) & (I < 0.43)] = 1
policy[(R < 0.34) & (I > 0.55)] = 2
policy[(R >= 0.34) & (I >= 0.45)] = 3
policy[(R > 0.73) & (I > 0.75)] = 4

cmap = ListedColormap(["#440154", "#FDE725", "#35B779", "#31688E", "#21918C"])
norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(
    policy,
    origin="lower",
    extent=[reserve.min(), reserve.max(), imbalance.min(), imbalance.max()],
    aspect="auto",
    cmap=cmap,
    norm=norm
)
ax.set_title("Cluster optimization policy map", pad=16)
ax.set_xlabel("reserve capacity")
ax.set_ylabel("load imbalance")

labels = [
    (0.22, 0.25, "steady"),
    (0.72, 0.25, "reserve"),
    (0.22, 0.72, "shed /\nshorten"),
    (0.55, 0.61, "rebalance"),
    (0.83, 0.86, "scale-out"),
]
for x, y, text in labels:
    ax.text(x, y, text, ha="center", va="center", fontsize=18)

cbar = fig.colorbar(im, ax=ax, ticks=[0,1,2,3,4])
cbar.ax.set_yticklabels(["steady", "reserve", "shed/shorten", "rebalance", "scale-out"])
cbar.set_label("selected cluster policy")

savefig("59_cluster_policy_map.png")

## 3. Global optimization loop

The cluster optimizer repeatedly observes, aggregates, evaluates, optimizes, broadcasts, and executes.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Global optimization loop", pad=24)

steps = [
    ("Observe", 0.50, 0.86),
    ("Aggregate\ncluster state", 0.78, 0.68),
    ("Evaluate\nobjective", 0.75, 0.36),
    ("Optimize\nplacement", 0.50, 0.18),
    ("Broadcast\npolicy", 0.25, 0.36),
    ("Local\nexecution", 0.22, 0.68),
]
for label, x, y in steps:
    rounded_box(ax, (x-0.11, y-0.055), 0.22, 0.11, label, fontsize=16)

for i in range(len(steps)):
    _, x1, y1 = steps[i]
    _, x2, y2 = steps[(i+1) % len(steps)]
    arrow(ax, (x1, y1-0.07 if y2 < y1 else y1+0.07), 
          (x2, y2+0.07 if y2 < y1 else y2-0.07),
          connectionstyle="arc3,rad=0.15")

ax.text(
    0.5, 0.50,
    r"$\max_{\pi}\; U(\pi)=G(\pi)-L(\pi)-M(\pi)-S(\pi)-V(\pi)$",
    ha="center", va="center", fontsize=22
)

ax.text(
    0.5, 0.04,
    "Cluster policy is not a static rule; it is an updated control loop.",
    ha="center", fontsize=16
)

savefig("59_global_optimization_loop.png")

## 4. Replica placement policy

A cluster-level policy allocates draft, verification, reserve, and telemetry capacity across heterogeneous nodes.

In [ ]:
nodes = ["node A", "node B", "node C", "node D", "node E", "node F"]
components = ["draft replicas", "verification replicas", "standby replicas", "telemetry"]
values = np.array([
    [0.50, 0.22, 0.18, 0.10],
    [0.32, 0.38, 0.20, 0.10],
    [0.40, 0.25, 0.25, 0.10],
    [0.24, 0.30, 0.36, 0.10],
    [0.55, 0.16, 0.19, 0.10],
    [0.28, 0.36, 0.26, 0.10],
])

fig, ax = plt.subplots(figsize=(15, 9))
left = np.zeros(len(nodes))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"][:4]

for i, comp in enumerate(components):
    ax.barh(nodes, values[:, i], left=left, label=comp, color=colors[i])
    for j, node in enumerate(nodes):
        if values[j, i] > 0.13:
            ax.text(left[j] + values[j, i]/2, j, f"{comp.split()[0]}\n{values[j,i]:.2f}",
                    ha="center", va="center", fontsize=12)
    left += values[:, i]

ax.set_xlim(0, 1)
ax.set_xlabel("placement fraction")
ax.set_title("Replica placement policy across heterogeneous nodes", pad=18)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.18), ncol=4)
ax.grid(axis="x", alpha=0.3)

savefig("59_replica_placement_policy.png")

## 5. Cluster convergence

Cluster optimization should reduce persistent imbalance rather than only raise throughput.

In [ ]:
steps = np.arange(0, 31, 3)
target = 0.72
initials = np.array([0.92, 0.51, 0.81, 0.62, 0.38, 0.76])
rates = np.array([0.22, 0.20, 0.18, 0.17, 0.25, 0.14])

fig, ax = plt.subplots(figsize=(14, 8))
for i, (u0, rate) in enumerate(zip(initials, rates), start=1):
    curve = target + (u0 - target) * np.exp(-rate * steps)
    ax.plot(steps, curve, marker="o", label=f"node {i}")

ax.axhline(target, color="black", linestyle="--", label="cluster target")
ax.set_title("Node utilization balancing across the cluster", pad=18)
ax.set_xlabel("coordination step")
ax.set_ylabel("GPU utilization")
ax.set_ylim(0.3, 1.02)
ax.grid(alpha=0.3)
ax.legend(ncol=3, loc="upper right")

savefig("59_cluster_convergence.png")

## 6. Failure recovery policy

Recovery is a controlled path: local fallback protects service while global capacity shifts restore the cluster.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Failure recovery policy", pad=22)

main = [
    ("node\nhealthy", 0.05, 0.56),
    ("overload /\nfailure", 0.25, 0.56),
    ("telemetry\nalert", 0.45, 0.56),
    ("traffic\nreroute", 0.65, 0.56),
    ("recover /\nrebalance", 0.84, 0.56),
]
for label, x, y in main:
    rounded_box(ax, (x, y), 0.15, 0.12, label, fontsize=15)
for i in range(len(main)-1):
    _, x1, y1 = main[i]
    _, x2, y2 = main[i+1]
    arrow(ax, (x1+0.15, y1+0.06), (x2, y2+0.06))

rounded_box(ax, (0.40, 0.26), 0.25, 0.12, "fallback:\nlocal shorten", fontsize=15)
rounded_box(ax, (0.68, 0.26), 0.25, 0.12, "global:\ncapacity shift", fontsize=15)

arrow(ax, (0.525, 0.56), (0.525, 0.38))
arrow(ax, (0.65, 0.32), (0.68, 0.32))
arrow(ax, (0.805, 0.38), (0.725, 0.56), connectionstyle="arc3,rad=-0.25")

ax.text(
    0.5, 0.07,
    "Cluster optimization keeps local failures from becoming cluster-wide failures.",
    ha="center", fontsize=17
)

savefig("59_failure_recovery_policy.png")

## 7. Objective decomposition

The optimizer balances throughput gain against latency pressure, migration cost, spillover risk, and verification cost.

In [ ]:
x = np.linspace(0, 1, 240)
throughput = 1.25 * (1 - np.exp(-3.2 * x))
latency = 0.20 + 1.05 * x**3
spillover = 0.14 + 0.28 * np.maximum(x - 0.62, 0)**2 * 6
migration = 0.10 * x + 0.08 * x**2
verification = 0.09 + 0.10 * np.sin(np.pi * x)**2
objective = throughput - 0.45*latency - 0.35*spillover - 0.45*migration - 0.25*verification

best_idx = np.argmax(objective)
best_x = x[best_idx]

fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(x, throughput, label="throughput benefit")
ax.plot(x, latency, label="latency pressure")
ax.plot(x, spillover, label="failure spillover risk")
ax.plot(x, objective, label="cluster objective", linewidth=2.5)
ax.axvline(best_x, color="black", linestyle="--", label=f"best ≈ {best_x:.2f}")

ax.set_title("Cluster objective balances gain, latency, and spillover", pad=18)
ax.set_xlabel("global optimization intensity")
ax.set_ylabel("normalized value")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")

savefig("59_cluster_objective_decomposition.png")

## 8. Complete cluster architecture

This is the capstone systems view: users, frontends, regional schedulers, global optimizer, replica placement, workers, telemetry.

In [ ]:
fig, ax = plt.subplots(figsize=(17, 10))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Complete adaptive serving architecture", pad=24)

rounded_box(ax, (0.06, 0.75), 0.16, 0.10, "users", fontsize=16)
rounded_box(ax, (0.30, 0.75), 0.18, 0.10, "frontends", fontsize=16)
rounded_box(ax, (0.58, 0.75), 0.20, 0.10, "regional\nschedulers", fontsize=16)

rounded_box(ax, (0.38, 0.50), 0.24, 0.11, "global\noptimizer", fontsize=17)
rounded_box(ax, (0.08, 0.35), 0.20, 0.10, "replica\nplacement", fontsize=16)
rounded_box(ax, (0.40, 0.30), 0.20, 0.10, "GPU\nworkers", fontsize=16)
rounded_box(ax, (0.72, 0.35), 0.20, 0.10, "verification\nworkers", fontsize=16)
rounded_box(ax, (0.34, 0.08), 0.32, 0.10, "telemetry fabric", fontsize=17)

arrow(ax, (0.22, 0.80), (0.30, 0.80))
arrow(ax, (0.48, 0.80), (0.58, 0.80))
arrow(ax, (0.68, 0.75), (0.56, 0.61))
arrow(ax, (0.50, 0.50), (0.18, 0.45), connectionstyle="arc3,rad=0.15")
arrow(ax, (0.50, 0.50), (0.50, 0.40))
arrow(ax, (0.50, 0.50), (0.82, 0.45), connectionstyle="arc3,rad=-0.15")

arrow(ax, (0.18, 0.35), (0.40, 0.35))
arrow(ax, (0.60, 0.35), (0.72, 0.40))
arrow(ax, (0.50, 0.30), (0.50, 0.18))
arrow(ax, (0.82, 0.35), (0.60, 0.18), connectionstyle="arc3,rad=-0.1")
arrow(ax, (0.50, 0.18), (0.50, 0.50), connectionstyle="arc3,rad=-0.25")

ax.text(
    0.5, 0.01,
    "The cluster optimizer translates telemetry into bounded placement, scaling, verification, and recovery policies.",
    ha="center", va="bottom", fontsize=16
)

savefig("59_complete_architecture.png")

## 9. Repository synthesis

Notebook 59 completes the second systems arc and opens the next arc: learning controllers.

In [ ]:
fig, ax = plt.subplots(figsize=(17, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Notebook 59 closes the second systems arc", pad=24)

# First arc
first = [
    ("00\nContext", 0.05),
    ("07\nVerification\nResource", 0.18),
    ("13\nConfidence\nScheduling", 0.31),
    ("17\nSemi-AR\nDrafting", 0.44),
    ("23\nThroughput\nObjective", 0.57),
    ("29\nHardware\nConstraints", 0.70),
    ("37\nOperating\nRegimes", 0.83),
]
y1 = 0.66
for label, x in first:
    rounded_box(ax, (x, y1), 0.10, 0.12, label, fontsize=11)
for i in range(len(first)-1):
    arrow(ax, (first[i][1]+0.10, y1+0.06), (first[i+1][1], y1+0.06), lw=1.8)

ax.text(0.50, 0.59, "first arc: mechanism → operating regime", ha="center", fontsize=16)
ax.plot([0.05, 0.94], [0.54, 0.54], color="black", lw=1.5)

# Second arc
second = [
    ("43\nResource\nAllocation", 0.27),
    ("49\nAdaptive\nInfrastructure", 0.42),
    ("53\nDistributed\nScheduling", 0.57),
    ("59\nCluster\nOptimization", 0.72),
]
y2 = 0.36
for label, x in second:
    rounded_box(ax, (x, y2), 0.12, 0.12, label, fontsize=11)
for i in range(len(second)-1):
    arrow(ax, (second[i][1]+0.12, y2+0.06), (second[i+1][1], y2+0.06), lw=1.8)

ax.text(
    0.50, 0.25,
    "second arc: controller → infrastructure → distributed scheduling → cluster optimization",
    ha="center", fontsize=16
)

# Third arc hint
rounded_box(ax, (0.38, 0.06), 0.24, 0.10, "next arc:\nlearning controllers", fontsize=14)
arrow(ax, (0.78, y2), (0.62, 0.11), connectionstyle="arc3,rad=-0.2")

savefig("59_repository_synthesis.png")

## Export

This cell zips the generated figures and provides a desktop-friendly download link when run in Colab/Jupyter.

In [ ]:
# Download cell — keep this in every notebook.
from pathlib import Path
import zipfile

zip_path = Path("59_cluster_optimization_v2_figures.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(FIG_DIR.glob("59_*.png")):
        zf.write(path, arcname=path.name)

print(f"Created {zip_path.resolve()}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(zip_path)))
    except Exception:
        print("Download link unavailable in this environment.")
        print(exc)

## Figure list

Generated files:

- `59_cluster_optimization_architecture.png`
- `59_cluster_policy_map.png`
- `59_global_optimization_loop.png`
- `59_replica_placement_policy.png`
- `59_cluster_convergence.png`
- `59_failure_recovery_policy.png`
- `59_cluster_objective_decomposition.png`
- `59_complete_architecture.png`
- `59_repository_synthesis.png`